Pipeline:


    1. Load RESISC45 (train / val / test) through TorchGeo. 

    2. Pass every image once through the frozen CLIP image encoder and cache the features. 

    3. Train a linear classifier on the cached features (fast, since the backbone is frozen). 
    
    4. Keep the epoch with the best validation accuracy and report test accuracy.

In [40]:
import argparse
import contextlib
from pathlib import Path
from torchvision.datasets import ImageFolder
import open_clip
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, random_split
import importlib
from tqdm import tqdm
from torchvision.transforms import ToTensor


In [55]:
# CLIP's pre-training normalization statistics (RGB).
CLIP_MEAN = (0.48145466, 0.4578275, 0.40821073)
CLIP_STD = (0.26862954, 0.26130258, 0.27577711)


def parse_args():
    p = argparse.ArgumentParser(description=__doc__, formatter_class=argparse.RawDescriptionHelpFormatter)
    p.add_argument("--root", default="data/resisc45", help="Dataset directory.")
    p.add_argument("--download", action="store_true", help="Download RESISC45 if missing.")
    p.add_argument("--model", default="ViT-B-32", help="open_clip model name.")
    p.add_argument("--pretrained", default="openai", help="open_clip pretrained tag.")
    p.add_argument("--cache-dir", default="cache", help="Where extracted features are stored.")
    p.add_argument("--extract-batch-size", type=int, default=256)
    p.add_argument("--workers", type=int, default=4)
    p.add_argument("--epochs", type=int, default=100)
    p.add_argument("--batch-size", type=int, default=512, help="Batch size for training the linear layer.")
    p.add_argument("--lr", type=float, default=1e-3)
    p.add_argument("--weight-decay", type=float, default=1e-2)
    p.add_argument("--seed", type=int, default=0)
    p.add_argument("--out", default="clip_linear_probe_resisc45.pt", help="Where to save the classifier.")
    return p.parse_args()


def preprocess(images, size, device):
    """Resize + normalize a batch on the GPU.

    TorchGeo's RESISC45 yields float images in [0, 255] with shape (B, 3, 256, 256).
    (If your TorchGeo version already returns [0, 1], drop the `/ 255.0`.)
    """
    x = images.to(device, non_blocking=True) / 255.0
    x = F.interpolate(x, size=size, mode="bicubic", align_corners=False, antialias=True).clamp(0, 1)
    mean = torch.tensor(CLIP_MEAN, device=device).view(1, 3, 1, 1)
    std = torch.tensor(CLIP_STD, device=device).view(1, 3, 1, 1)
    return (x - mean) / std


@torch.no_grad()
def extract_features(model, dataset, args, device):
    """Run the frozen CLIP image encoder over a dataset -> (features, labels)."""

    loader = DataLoader(
        dataset,
        batch_size=args.extract_batch_size,
        shuffle=False,
        num_workers=args.workers,
        pin_memory=device.type == "cuda",
    )

    size = model.visual.image_size
    size = (size, size) if isinstance(size, int) else tuple(size)

    feats, labels = [], []

    for batch in tqdm(loader, desc="Extracting features", leave=False):

        # Unpack images and labels
        images, batch_labels = batch

        # Preprocess images
        x = preprocess(images, size, device)

        # Extract CLIP image embeddings
        with torch.autocast("cuda", dtype=torch.float16) if device.type == "cuda" else contextlib.nullcontext():
            f = model.encode_image(x)

        # Normalize embeddings
        feats.append(F.normalize(f.float(), dim=-1).cpu())

        # Store corresponding labels
        labels.append(torch.as_tensor(batch_labels).cpu())

    return torch.cat(feats), torch.cat(labels)



def load_or_extract_features(args, device):
    cache = Path(args.cache_dir) / f"resisc45_{args.model}_{args.pretrained}.pt"
    if cache.exists():
        print(f"Loading cached features from {cache}")
        return torch.load(cache)

    model, _, _ = open_clip.create_model_and_transforms(args.model, pretrained=args.pretrained)
    model = model.to(device).eval().requires_grad_(False)  # frozen

    data = {}
    for split in ("train", "val", "test"):
        ds = RESISC45(root=args.root, split=split, download=args.download)
        data[split] = extract_features(model, ds, args, device)
        data["classes"] = ds.classes
        print(f"{split}: {len(ds)} images")

    cache.parent.mkdir(parents=True, exist_ok=True)
    torch.save(data, cache)
    return data


class LinearProbe(nn.Module):
    """Feature standardization (train-set statistics) followed by a single linear layer."""

    def __init__(self, in_dim, num_classes, mean, std):
        super().__init__()
        self.register_buffer("mean", mean)
        self.register_buffer("std", std)
        self.fc = nn.Linear(in_dim, num_classes)

    def forward(self, x):
        return self.fc((x - self.mean) / self.std)

@torch.no_grad()
def accuracy(clf, x, y, bs=4096):
    clf.eval()
    correct = sum((clf(x[i : i + bs]).argmax(1) == y[i : i + bs]).sum().item() for i in range(0, len(x), bs))
    return correct / len(x)


In [59]:
#data location

# ROOT = Path('NWPU-RESISC45/NWPU-RESISC45')
ROOT = Path('NWPU-RESISC45/NWPU-RESISC45')

dataset = ImageFolder(
    root=ROOT,
    transform=ToTensor()
)

print("Dataset location:", ROOT.resolve)
print("Images:", len(dataset))
print("Classes:", len(dataset.classes))
print(dataset.classes)

Dataset location: <bound method Path.resolve of WindowsPath('NWPU-RESISC45/NWPU-RESISC45')>
Images: 31494
Classes: 45
['airplane', 'airport', 'baseball_diamond', 'basketball_court', 'beach', 'bridge', 'chaparral', 'church', 'circular_farmland', 'cloud', 'commercial_area', 'dense_residential', 'desert', 'forest', 'freeway', 'golf_course', 'ground_track_field', 'harbor', 'industrial_area', 'intersection', 'island', 'lake', 'meadow', 'medium_residential', 'mobile_home_park', 'mountain', 'overpass', 'palace', 'parking_lot', 'railway', 'railway_station', 'rectangular_farmland', 'river', 'roundabout', 'runway', 'sea_ice', 'ship', 'snowberg', 'sparse_residential', 'stadium', 'storage_tank', 'tennis_court', 'terrace', 'thermal_power_station', 'wetland']


In [60]:
#splitting and loading the data

batch_size = 64

dataset_size = len(dataset)

# Split sizes
test_size = int(0.8 * dataset_size)
train_size = int(0.16 * dataset_size)
val_size = dataset_size - train_size - test_size

# Split dataset
train_dataset, val_dataset, test_dataset = random_split(
    dataset,
    [train_size, val_size, test_size],
    generator=torch.Generator().manual_seed(42)
)

train_loader = DataLoader(
    train_dataset, 
    batch_size=batch_size, 
    shuffle=True,
    num_workers=4, 
    pin_memory=True
)

x_tr, y_tr = map(torch.cat, zip(*train_loader))

val_loader = DataLoader(
    val_dataset, 
    batch_size=batch_size, 
    shuffle=False, 
    num_workers=4, 
    pin_memory=True
)

x_va, y_va = map(torch.cat, zip(*val_loader))

test_loader = DataLoader(
    test_dataset, 
    batch_size=batch_size, 
    shuffle=False, 
    num_workers=4, 
    pin_memory=True
)

x_te, y_te = map(torch.cat, zip(*val_loader))

print(f"Training dataset size: {len(train_dataset)}")
print(f"Validation dataset size: {len(val_dataset)}")

print(f"Test set size: {len(test_loader.dataset)}")

c:\Users\magnu\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\torch\utils\data\dataloader.py:1102: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Training dataset size: 5039
Validation dataset size: 1260
Test set size: 25195


In [61]:
from types import SimpleNamespace

args = SimpleNamespace(
    model="ViT-B-32",
    pretrained="openai",
    cache_dir=None,
    extract_batch_size=64,
    workers=0,
    epochs=5,
    batch_size=64,
    lr=1e-3,
    weight_decay=1e-4,
    seed=42,
    out="linear_probe.pt"
)

torch.manual_seed(args.seed)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

clip_model, _, _ = open_clip.create_model_and_transforms(
    args.model, pretrained=args.pretrained, cache_dir=args.cache_dir
)
clip_model = clip_model.to(device).eval()


x_tr, y_tr = extract_features(clip_model, train_dataset, args, device)
x_va, y_va = extract_features(clip_model, val_dataset, args, device)
x_te, y_te = extract_features(clip_model, test_dataset, args, device)


x_tr, y_tr, x_va, y_va, x_te, y_te = (
    t.to(device) for t in (x_tr, y_tr, x_va, y_va, x_te, y_te)
)
classes = dataset.classes


Device: cpu


c:\Users\magnu\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\open_clip\factory.py:555: UserWarning: QuickGELU mismatch between final model config (quick_gelu=False) and pretrained tag 'openai' (quick_gelu=True).
  warnings.warn(


KeyboardInterrupt: 

In [ ]:
clf = LinearProbe(x_tr.shape[1], len(classes), x_tr.mean(0), x_tr.std(0) + 1e-6).to(device)
opt = torch.optim.AdamW(clf.parameters(), lr=args.lr, weight_decay=args.weight_decay)
steps_per_epoch = (len(x_tr) + args.batch_size - 1) // args.batch_size
sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=args.epochs * steps_per_epoch)
i = 0
best_val, best_state = 0.0, None
for epoch in range(1, args.epochs + 1):

    clf.train()
    perm = torch.randperm(len(x_tr), device=device)
    for i in range(0, len(x_tr), args.batch_size):
        idx = perm[i : i + args.batch_size]
        loss = F.cross_entropy(clf(x_tr[idx]), y_tr[idx])
        opt.zero_grad(set_to_none=True)
        loss.backward()
        opt.step()
        sched.step()

    val_acc = accuracy(clf, x_va, y_va)
    if val_acc > best_val:
        best_val = val_acc
        best_state = {k: v.detach().clone() for k, v in clf.state_dict().items()}
    if epoch == 1 or epoch % 10 == 0:
        print(f"epoch {epoch:3d} | loss {loss.item():.4f} | val acc {val_acc:.4f}")

clf.load_state_dict(best_state)
test_acc = accuracy(clf, x_te, y_te)
print(f"\nBest val acc: {best_val:.4f} | Test acc: {test_acc:.4f}")

torch.save(
    {
        "state_dict": clf.state_dict(),
        "classes": classes,
        "clip_model": args.model,
        "pretrained": args.pretrained,
        "val_acc": best_val,
        "test_acc": test_acc,
    },
    args.out,
)
print(f"Saved classifier to {args.out}")





epoch   1 | loss 0.8991 | val acc 0.5000
epoch  10 | loss 0.0104 | val acc 0.5000
epoch  20 | loss 0.0015 | val acc 0.5000
epoch  30 | loss 0.0009 | val acc 0.5000
epoch  40 | loss 0.0008 | val acc 0.5000
epoch  50 | loss 0.0008 | val acc 0.5000

Best val acc: 0.5000 | Test acc: 0.6429
Saved classifier to linear_probe.pt
